# 🍜 頑固ラーメン屋のAIを作ろう (Build a Stubborn Ramen AI)

---
```text
 FFFFF   A   QQQQ      BBBB   OOO  TTTTT
 F      A A  Q  Q      B  B  O   O   T
 FFF   AAAAA Q  Q      BBBB  O   O   T
 F     A   A Q  Q      B  B  O   O   T
 F     A   A  QQ Q     BBBB   OOO    T
```
# 📗 ラボ1（午前）：初めての言語モデル構築
### (Lab 1: Your First Language Model — Bigram)
---

このラボでは、Pythonを使って **頑固なラーメン屋の店長** のようなAIをゼロから作ります。
最初は何も知らないAIですが、学習（トレーニング）させることで、ラーメン屋らしい会話ができるようになります。

# 📖 今日の辞書 (Concept Dictionary)
難しい言葉が出てきたら、ここを見てください。

| 言葉 (Term) | 意味 (Meaning) | 例え (Analogy) |
| :--- | :--- | :--- |
| **Tokens (トークン)** | AIが読む文字の単位 | 「あ」「い」「う」など、文字ひとつひとつ |
| **Vocabulary (語彙)** | AIが知っている全文字のリスト | 五十音表＋漢字リスト |
| **Batch Size (バッチサイズ)** | 一度に勉強する量 | 英単語カードを「32枚ずつ」めくる |
| **Loss (ロス・損失)** | AIの間違いの大きさ | テストの「バツ」の数。**0に近いほど賢い！** |
| **Optimizer (オプティマイザ)** | AIを修正する先生 | 「そこ間違ってるよ、こう直しなさい」と指導する役割 |
| **Iteration (イテレーション)** | 学習の回数 | ドリルを繰り返す回数 |
| **Overfitting (過学習)** | 丸暗記状態 | 練習問題は満点なのに、初見の問題が解けない |


# ==============================================================================
# ✅ パート０：ライブラリの読み込み
# ==============================================================================
ニューラルネットワークの構築には **PyTorch** を、学習の進み具合の可視化には **Matplotlib** を使用します。


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)  # 不要な警告を非表示

import torch                          # PyTorch本体
import torch.nn as nn                 # ニューラルネットワークの部品集
from torch.nn import functional as F  # 損失関数などのよく使う関数集
import matplotlib.pyplot as plt       # グラフ描画
import japanize_matplotlib            # グラフの日本語表示

# 実行のたびに同じ結果になるよう、乱数の種を固定します（再現性の確保）
torch.manual_seed(1337)
print("ライブラリ読み込み完了！")

# ==============================================================================
# ✅ パート1：教科書（データ）の準備
# ==============================================================================

AIに読ませる「頑固ラーメン屋の会話集」です。
AIはこのテキストの **パターンを真似する** ように学習します。

> 💡 **ポイント**: モデルにとってデータは、人間にとっての教科書と同じです。
> データの質と量が、最終的なモデルの性能を大きく左右します。


In [ ]:
text = """
客：いらっしゃい。
店：へい、いらっしゃい。食券買ってな。
客：おすすめは何ですか？
店：うちは塩ラーメンしか置いてないよ。メニューをよく見なさい。
客：すいません、お水ください。
店：水はセルフサービスだよ。あそこの給水機を使ってくれ。
客：大盛りはできますか？
店：うちは大盛りやってないんだ。味のバランスが崩れるからね。
客：麺の硬さは選べますか？
店：うちは「普通」が一番うまいんだ。黙って座って待ってな。
客：ごちそうさまでした。
店：おう、まいど。丼はカウンターに上げてってな。
客：トイレはどこですか？
店：店の外を出て右だよ。
客：替え玉お願いします。
店：だから、メニュー見てくれよ。替え玉もやってないんだ。
""" * 100  # データ量を増やすために100回繰り返します

print(f"データの文字数: {len(text)} 文字")

## 「文字」を「数字」に変換する（前処理・トークン化）

コンピュータは人間の言葉をそのまま理解できません。**AIは計算しかできない** ため、
文字を数字に変換する必要があります。この工程を **前処理 (Pre-processing)**、
変換された数字を **トークン (Token)** と呼びます。

手順は3ステップです：

1. **語彙の作成**: データに含まれるすべてのユニークな文字を集めてリスト化する
2. **対応表の作成**: 各文字に一意の番号（ID）を割り当てる（`あ→4`, `い→5` ...）
3. **変換関数の作成**: 文字列⇄数字列を相互変換する `encode` / `decode` を作る

```text
 エンコード:  "おすすめ"  ──(stoi辞書)──▶  [9, 19, 19, 37]
 デコード  :  [9, 19, 19, 37]  ──(itos辞書)──▶  "おすすめ"
```


In [ ]:
### データの中にどんな文字があるかリストアップします
# set()   : 重複を除いたユニークな文字の集合を作る
# sorted(): 文字コード順に並べ替え → 実行のたびに同じ対応表になる（再現性）
chars = sorted(list(set(text)))
vocab_size = len(chars)   # 語彙数。後でモデルの出力サイズを決めるのに使います

print(f"使われている文字の種類: {vocab_size} 種類")
print(f"文字リスト: {''.join(chars)}")

### 文字と数字の対応表を作ります (Encoding / Decoding)
stoi = { ch:i for i,ch in enumerate(chars) }  # string-to-integer: 文字 → 番号
itos = { i:ch for i,ch in enumerate(chars) }  # integer-to-string: 番号 → 文字
encode = lambda s: [stoi[c] for c in s]           # 文字列 → 数字リスト
decode = lambda l: ''.join([itos[i] for i in l])  # 数字リスト → 文字列

print("\n--- 変換テスト ---")
print(f"元の言葉: おすすめ")
print(f"AIの視点: {encode('おすすめ')}")
print(f"復元    : {decode(encode('おすすめ'))}")

## データのテンソル化と、訓練用/検証用への分割

### なぜ「一度だけ」変換するのか？
テキスト全体を **一度だけ** 数字（テンソル）に変換して保存しておきます。
学習中に何千回もデータを取り出しますが、毎回変換し直すのは無駄だからです。
（料理に例えると：使うたびに米を精米するのではなく、最初にまとめて精米しておくイメージ）

### なぜ訓練用と検証用に分けるのか？
学生の試験勉強に例えると：

| データ | 例え | 役割 |
| :--- | :--- | :--- |
| **訓練データ (train) 90%** | 教科書・練習問題 | モデルがパターンを学ぶ。パラメータ更新に使う |
| **検証データ (val) 10%** | 模擬試験 | 学習には使わず、「初見の問題への実力」を測る |

学習中に両方の損失（Loss）を見ることで、
**「練習問題を丸暗記しているだけではないか？（過学習）」** を監視できます。

> ⚠️ **正直な注意書き**: 今日のデータは同じ会話を100回繰り返したものなので、
> 訓練データと検証データの中身は実質同じです。そのため今日は両者の損失がほぼ同じ値で
> 下がっていきます。**実際のプロジェクトでは多様なデータを使うため、
> 「訓練Lossだけ下がり、検証Lossが上がり始めたら過学習」** というサインになります。
> この「作法」を今日から身につけておきましょう。


In [ ]:
# テキスト全体を一度だけテンソルに変換します
# dtype=torch.long : 64ビット整数。埋め込み層(Embedding)が要求するデータ型です
data = torch.tensor(encode(text), dtype=torch.long)
print(f"データ全体のテンソル形状: {data.shape}")
print(f"最初の20トークン: {data[:20].tolist()}")

# 前から90%を訓練用、残り10%を検証用に分割します
n = int(0.9 * len(data))
train_data = data[:n]   # スライス記法: 最初からn番目まで
val_data   = data[n:]   # n番目から最後まで
print(f"訓練データ: {len(train_data)} トークン / 検証データ: {len(val_data)} トークン")

# ==============================================================================
# ✅ パート2：AIの「脳」を構築する
# ==============================================================================
**※ このセルは変更しなくてOKです。実行 (Shift+Enter) だけしてください。**

## Bigramモデルとは？
今日の最初のモデルは **Bigram（バイグラム）モデル** — 「直前の1文字だけ」を見て次の文字を予測する、
最もシンプルな言語モデルです。

```text
 「ラ」を見る → 表を引く → 「ー」の確率が高い！ → 「ー」を出力
```

中身は、たった1枚の **巨大な確率表**（`nn.Embedding`）です。
「文字Aの次には文字Bが来やすい」という統計を学習します。

## バッチ (Batch) — 教材の小分け
データ全体を一度に学習させるのではなく、ランダムに小さな塊（バッチ）を取り出して学習します。
- **効率性**: メモリ使用量を抑えられる
- **安定性**: ランダムな順序で学ぶことで、偏った学習（過学習）を防げる

`get_batch` は「入力x」と「正解y（xを1文字ずらしたもの）」のペアを32個ずつ取り出します。


In [ ]:
block_size = 8   # モデルが一度に見る文字数（コンテキスト長）
batch_size = 32  # 一度に学習する例の数

class BigramLanguageModel(nn.Module):
    """直前の1文字だけを見て次の文字を予測する、最小の言語モデル"""

    def __init__(self, vocab_size):
        super().__init__()
        # 各トークン（文字）ごとに「次の文字の確率スコア表」を持つテーブル
        # サイズ: (語彙数 × 語彙数) — つまり全文字の組み合わせの統計表
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx: 入力トークン列 (バッチ, 文字数)
        # 表を引くだけで、各位置の「次の文字スコア(logits)」が得られる
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None   # 生成時（正解がない場合）は損失を計算しない
        else:
            # 予測(logits)と正解(targets)を比べて、誤差(loss)を計算する
            # cross_entropy は「正解の文字にどれだけ高い確率を付けられたか」を測る損失関数
            B, T, C = logits.shape            # B=バッチ, T=文字数, C=語彙数
            logits = logits.view(B*T, C)      # 損失計算のために形を変える
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # 現在の文字列から、次の文字を1つずつ予測して繋げていくループ
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]                        # 最後の文字の予測だけ使う
            probs = F.softmax(logits, dim=-1)                # スコア → 確率分布に変換
            idx_next = torch.multinomial(probs, num_samples=1)  # 確率に従って1文字抽選
            idx = torch.cat((idx, idx_next), dim=1)          # 文脈に追加して繰り返す
        return idx

# データをバッチ（小分け）にする関数
def get_batch(split):
    # split の値 ('train' / 'val') に応じてデータを選択
    d = train_data if split == 'train' else val_data
    # バッチとして取り出す開始位置を batch_size 個ランダムに選ぶ
    ix = torch.randint(len(d) - block_size, (batch_size,))
    # 入力x: 各開始位置から block_size 文字
    x = torch.stack([d[i:i+block_size] for i in ix])
    # 正解y: xを1文字だけ右にずらしたもの（「次の文字」が正解だから）
    y = torch.stack([d[i+1:i+block_size+1] for i in ix])
    return x, y

# 学習中に train / val 両方の損失を測る関数
@torch.no_grad()          # 評価中は勾配計算が不要（メモリと時間の節約）
def estimate_loss(model, eval_iters=100):
    out = {}
    model.eval()          # 評価モードに切り替え
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()   # 何回か測って平均する（測定のブレを減らすため）
    model.train()         # 訓練モードに戻す
    return out

print("AIの脳の「設計図」が完成しました！")

# ==============================================================================
# 🧐 先生用の解説コーナー (Instructor Demo)
# ==============================================================================
学習を始める前に、AIが何を「正解」としているか確認しましょう。

言語モデルの教材は、**「ある文字を見て、その次の文字を当てる」** クイズの繰り返しです。


In [ ]:
x_demo, y_demo = get_batch('train')

print("例：ラーメン屋のデータの「入力」と「正解」の関係\n")

# 最初のバッチの、最初の4文字だけを見てみます
for i in range(4):
    input_text  = decode([x_demo[0, i].item()])
    target_text = decode([y_demo[0, i].item()])

    print(f"ステップ {i+1}:")
    print(f"  AIが見ている文字 (Input):  '{input_text}'")
    print(f"  AIが当てるべき文字 (Target): '{target_text}'")
    print("-------------------------")

# ==============================================================================
# 🔧 パート3：AIのトレーニング (★ ここが今日の主役です！)
# ==============================================================================

ここから皆さんの出番です。
以下のコードには **空欄（`____________________`）** があります。
ヒントを読みながら、正しいコードや数字を入れてAIを学習させてください。

## トレーニングループの仕組み
1反復（イテレーション）で起こることは、いつも同じ4ステップです：

```text
 ┌──────────────────────────────────────────────────┐
 │ 1. フォワードパス : モデルが予測し、誤差(Loss)を計算  │
 │ 2. 勾配の初期化   : 前回の計算のゴミを掃除            │
 │ 3. バックワードパス: 間違いの原因（勾配）を逆算        │
 │ 4. パラメータ更新 : オプティマイザが少しだけ修正       │
 └──────────────────── ×何千回 ─────────────────────┘
```

**オプティマイザ**は「霧の中で山を下るガイド」です。現在地の傾斜（勾配）を見て、
損失という谷底へ向かう一歩を教えてくれます。歩幅が **学習率 (lr)** です。


In [ ]:
# モデルとオプティマイザを初期化します
model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
# AdamW: 現在最も広く使われる高性能オプティマイザ
# lr=1e-3 (=0.001): 学習率。大きすぎると学習が暴れ、小さすぎると遅い

# 損失の記録用リスト（後でグラフにします）
history = {"iter": [], "train": [], "val": []}

# -----------------------------------------------------------------
# 実験コーナー：パラメータを設定しよう (Fill in the blanks!)
# -----------------------------------------------------------------

# 学習回数 (AIがドリルを解く回数)
# ヒント: 100回だと全然足りませんが、3000回やると少し賢くなります。
# 推奨: 最初は 100 で試して、後で 3000 に書き換えてみましょう。
max_iters = ____________________


print(f"{max_iters}回の学習を開始します...")

for iter in range(max_iters):
    # データを取得
    xb, yb = get_batch('train')

    # -------------------------------------------------------------
    # クイズ：学習ループを完成させよう (ヒントを見て選んでください)
    # -------------------------------------------------------------

    # 1. 誤差(Loss)を計算する
    logits, loss = model(xb, yb)

    # 2. 前回の計算の「ゴミ」を掃除する呪文
    # ヒント: [ 選択肢: optimizer.zero_grad(set_to_none=True)  または  print("hello") ]
    # -------------------------------------------------------------
    ____________________

    # 3. バックプロパゲーション (間違いの原因を探る)
    loss.backward()

    # 4. パラメータを実際に更新する（先生が修正する）呪文
    # ヒント: [ 選択肢: optimizer.step()  または  optimizer.stop() ]
    # -------------------------------------------------------------
    ____________________

    # 500回ごとに、訓練/検証の両方で成績（Loss）を測って記録します
    if iter % 500 == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        history["iter"].append(iter)
        history["train"].append(losses['train'].item())
        history["val"].append(losses['val'].item())
        print(f"回数: {iter}, 訓練Loss: {losses['train']:.4f}, 検証Loss: {losses['val']:.4f}")

print("学習完了！")

## 📉 学習曲線を見てみよう (Loss Curve)

数字の羅列よりグラフの方が一目瞭然です。**右肩下がりなら学習成功** です。

> 👀 **見るポイント**:
> - 訓練Lossと検証Lossが **両方** 下がっていれば順調
> - もし訓練Lossだけ下がり検証Lossが上がり始めたら **過学習** のサイン
>   （今日のデータでは起きません — パート1の注意書きを参照）


In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history["iter"], history["train"], marker="o", label="訓練 Loss (train)")
plt.plot(history["iter"], history["val"],   marker="s", label="検証 Loss (val)")
plt.title("学習曲線：Lossが下がる = AIが賢くなっている")
plt.xlabel("学習回数 (Iteration)")
plt.ylabel("Loss（間違いの大きさ）")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# ==============================================================================
# ✅ パート4：AI店長と話してみよう
# ==============================================================================
何もない状態から300文字を生成させます。どんな「日本語」が出てくるでしょうか？


In [ ]:
print("--- AI店長の生成テキスト ---")

# 何もない状態(トークン0)からスタートして、300文字生成させます
context = torch.zeros((1, 1), dtype=torch.long)
generated_output = model.generate(context, max_new_tokens=300)[0].tolist()

# 数字を文字に戻して表示
print(decode(generated_output))

# ==============================================================================
# 🎮 パート5：リアルタイムで会話してみよう (Interactive Chat)
# ==============================================================================
「終了」と入力するとループを抜けます。

> 💡 コツ: 「客：」から書き始めると、AIが「店：」と答えやすくなります。


In [ ]:
import sys

def chat_with_ai(start_text):
    # 1. ユーザーの入力を数字(トークン)に変換
    try:
        context = torch.tensor([encode(start_text)], dtype=torch.long)
    except KeyError:
        # 学習データに存在しない文字は変換できません（語彙にないため）
        print("エラー: 学習データにない文字が含まれています。")
        return

    # 2. AIに続きを書かせる (50文字くらい)
    print(f"あなた: {start_text}", end="")
    sys.stdout.flush()

    generated_indices = model.generate(context, max_new_tokens=50)[0].tolist()

    # 3. 数字を文字に戻す (入力した部分はカットして、AIの生成部分だけ表示)
    full_text = decode(generated_indices)
    ai_response = full_text[len(start_text):]

    print(f"AI店長: {ai_response}")
    print("-" * 30)

print("AI店長と会話できます。「終了」と打つと終わります。")
print("コツ: 「客：」から書き始めると、AIが「店：」と答えやすくなります。")
print("-" * 30)

while True:
    user_input = input("入力してください (例: 客：おすすめは？) >> ")
    if user_input == "終了":
        break
    chat_with_ai(user_input)

<br>
<hr>

# 🆘 お助けコーナー (Rescue Cell / Answer Key)

もしエラーが出て動かなくなっても大丈夫です！
下のコードは**正解**です。コピーして「パート3」のセルに貼り付けてください。

```python
# 🆘 正解コード (Answer Key)

model = BigramLanguageModel(vocab_size)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
history = {"iter": [], "train": [], "val": []}

max_iters = 3000  # 学習回数は多いほうが賢くなります

print(f"{max_iters}回の学習を開始します...")

for iter in range(max_iters):
    xb, yb = get_batch('train')

    # 1. 誤差計算
    logits, loss = model(xb, yb)

    # 2. 掃除 (答え: optimizer.zero_grad)
    optimizer.zero_grad(set_to_none=True)

    # 3. 逆伝播
    loss.backward()

    # 4. 更新 (答え: optimizer.step)
    optimizer.step()

    if iter % 500 == 0 or iter == max_iters - 1:
        losses = estimate_loss(model)
        history["iter"].append(iter)
        history["train"].append(losses['train'].item())
        history["val"].append(losses['val'].item())
        print(f"回数: {iter}, 訓練Loss: {losses['train']:.4f}, 検証Loss: {losses['val']:.4f}")

print("学習完了！")
```


<br>
<hr>

# 🎉 午前の部：終了！ (Morning Session Complete)

お疲れ様でした！これで「初めての言語モデル」の構築は完了です。

### ✅ 午前中に学んだこと (Key Takeaways)
1.  **AIは魔法ではない**: 巨大な「確率の計算機」であり、次に来る文字を予測しているだけです。
2.  **学習のサイクル**: `予測` → `間違い(Loss)を計算` → `修正(Optimizer)` を何千回も繰り返すことで賢くなります。
3.  **Loss (損失)**: グラフが右肩下がりになるほど、AIはラーメン屋の口調を真似できるようになりました。
4.  **訓練/検証データ**: 「丸暗記（過学習）」を見張るための、機械学習の基本作法を学びました。

---

### 🤔 でも、AIの頭が悪くないですか？ (The Problem)
今のAI（Bigramモデル）は、単語の綴りは覚えたようですが、会話の内容はまだ支離滅裂です。

**なぜでしょうか？**

それは、このAIが **「金魚 (Goldfish)」** だからです。
記憶力が「直前の1文字」しかありません。「いらっしゃい」と言うために、「い」だけを見て「ら」を予測しています。前の会話なんて忘れています。

### 🚀 午後の予告：脳の移植手術 (Coming Up Next)
お昼休憩のあと、このAIに **「脳の移植手術」** を行います。

午後の部では、GoogleやOpenAIが採用している技術 **「Transformer (トランスフォーマー)」** を導入します。
AIに「記憶力（文脈を読む力）」を与えると、会話がどう劇的に変わるのか...お楽しみに！

では、お昼休みに入りましょう！🍜
